In [ ]:
import os
from TFBS_negatives.data import DataModule
from TFBS_negatives.models import TFmodel
import pytorch_lightning as pl
import pandas as pd
import torch
from tqdm import tqdm

output_folder = "/data/home/natant/Negatives/Runs/Review_sane_model/Cross_TF"
input_folder = "/data/home/natant/Negatives/Runs/Review_sane_model/other_cell_lines/"
os.makedirs(output_folder, exist_ok=True)


ckpt_files = [f for f in os.listdir(input_folder) if f.endswith('.ckpt')]

cell_type_to_check = 'A549'
cross_val_sets = ['CV-0', 'CV-1', 'CV-2', 'CV-3', 'CV-4', 'CV-5']


neg_mode_to_check_list = ['shuffled', 'dinucl_sampled', 'dinucl_shuffled', "celltype"]

for cross_val_set in cross_val_sets:
    for neg_mode_to_check in neg_mode_to_check_list:
        print(f"Processing negative mode: {neg_mode_to_check}")
        selected_files = []
        TF_list = []
        target_files = {}
        for file_name in ckpt_files:
            file_name_temp = file_name.split('.ckpt')[0]
            parts = file_name_temp.split('_')

            # Extract fields from the new naming scheme:
            # CT-GM12878, TF-ELK1$(1277-1), NEG-dinucl$shuffled, CV-5, ...
            cellline = None
            tf = None
            neg_type = None
            cv_split = None

            for p in parts:
                if p.startswith('CT-'):
                    celltype = p[len('CT-'):]
                elif p.startswith('TF-'):
                    # restore original underscores inside TF name
                    tf = p[len('TF-'):].replace('$', '_')
                elif p.startswith('NEG-'):
                    # keep '-' for later convert_dict mapping, but restore '_' inside mode
                    neg_mode = p[len('NEG-'):].replace('$', '_')
                elif p.startswith('CV-'):
                    CV = p[len('CV-'):]

            
            if neg_mode == neg_mode_to_check and celltype == cell_type_to_check and f'CV-{str(CV)}' == cross_val_set:
                selected_files.append(file_name)
                if TF not in TF_list:
                    target_files[TF] = file_name
                    TF_list.append(TF)
        if neg_mode_to_check == 'celltype':
            data_file = f"/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr copy/{cell_type_to_check}.h5t"
        else:
            data_file = f"/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/{cell_type_to_check}.h5t"
        results_dict_AUROC = {}
        results_dict_AUROC_HQ = {}

        for target_TF in tqdm(TF_list):
            file_name = target_files[target_TF]
            print(f"Processing file: {file_name}")
            results_dict_AUROC[target_TF] = []
            results_dict_AUROC_HQ[target_TF] = []
            file = input_folder + file_name
            
            
            for TF in TF_list:
                best_model = TFmodel.load_from_checkpoint(file)
                trainer = pl.Trainer(
                        accelerator="gpu",
                        devices=[3]
                    )
                Dmod = DataModule(data_file, TF=TF, batch_size=256, neg_mode=neg_mode_to_check, cross_val_set=int(cross_val_set.split('-')[1]))
                test_out = trainer.test(best_model, datamodule=Dmod)
                results_dict_AUROC[target_TF].append(test_out[0]['test_AUROC'])
                results_dict_AUROC_HQ[target_TF].append(test_out[0]['test_AUROC_HQ'])
                print(f"Model TF: {target_TF}, Test TF: {TF}, AUROC: {test_out[0]['test_AUROC']}, AUROC_HQ: {test_out[0]['test_AUROC_HQ']}")

                del best_model
                del trainer
                del Dmod
                import gc; gc.collect()
                torch.cuda.empty_cache()
        df_auroc = pd.DataFrame(results_dict_AUROC, index=TF_list)
        df_auroc_hq = pd.DataFrame(results_dict_AUROC_HQ, index=TF_list)

        # df_auroc.to_csv(os.path.join(output_folder, f"{cell_type_to_check}_{neg_mode_to_check}_{cross_val_set}_AUROC.csv"))
        # df_auroc_hq.to_csv(os.path.join(output_folder, f"{cell_type_to_check}_{neg_mode_to_check}_{cross_val_set}_AUROC_HQ.csv"))

 

/data/home/natant/anaconda3/envs/Negs2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing negative mode: shuffled


0it [00:00, ?it/s]


Processing negative mode: dinucl_sampled


0it [00:00, ?it/s]


...

0it [00:00, ?it/s]


Processing negative mode: dinucl_shuffled


0it [00:00, ?it/s]


Processing negative mode: celltype


0it [00:00, ?it/s]


In [9]:
f'CV-{str(CV)}'

'CV-4'